In [28]:
from kaggle_environments import make
from kinds import TileKind, CropType, AnimalType, AnimalStructureTile, parse_tile

from pathlib import Path
import json

json_path = Path("/home/doruk_sarigun/Çalışmalar/Kaggricalture/game_observations/game_04_seed_404.json")
with json_path.open("r") as f:
    json_file = json.load(f)

#json_file["steps"][i]["players"][0]["observation"]["farms"][0]["tiles"]

env = make("kaggriculture", configuration={"episodeSteps": 100})

In [29]:
obs =env.reset()

In [30]:
tile_rows = json_file["steps"][150]["players"][1]["observation"]["farms"][0]["tiles"]
tile_rows

[[None,
  None,
  None,
  None,
  None,
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 [None,
  None,
  {'kind': 'WEED'},
  None,
  None,
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 [None,
  None,
  {'kind': 'WEED'},
  None,
  None,
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 [None,
  None,
  {'kind': 'WEED'},
  None,
  {'kind': 'PLANT',
   'crop': 'TOMATO',
   'planted_day': 6,
   'watered_today': False,
   'consecutive_unwatered': 1,
   'yield_units': 0,
   'max_lifespan_step': -1,
   'fertilized_until_day': -1},
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 [None,
  None,
  None,
  None,
  {'kind': 'WEED'},
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 ['LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 ['LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 ['LOCKED',


In [31]:
for index, kind in enumerate(TileKind):
    print(index, kind)

0 EMPTY
1 LOCKED
2 WEED
3 PLANT
4 COOP
5 PASTURE


In [32]:
tile_categories = {kind: index for index, kind in enumerate(TileKind)}
crop_categories = {None: 0, **{crop: index for index, crop in enumerate(CropType, start=1)}}
animal_categories = {None: 0, **{animal: index for index, animal in enumerate(AnimalType, start=1)}}

In [33]:
tile_categories

{<TileKind.EMPTY: 'EMPTY'>: 0,
 <TileKind.LOCKED: 'LOCKED'>: 1,
 <TileKind.WEED: 'WEED'>: 2,
 <TileKind.PLANT: 'PLANT'>: 3,
 <TileKind.COOP: 'COOP'>: 4,
 <TileKind.PASTURE: 'PASTURE'>: 5}

In [34]:
animal_categories[None]

0

In [35]:
raw_tile = {
    "kind": "PLANT",
    "crop": "MELON",
    "planted_day": 4,
    "watered_today": False,
    "consecutive_unwatered": 1,
    "yield_units": 1,
    "max_lifespan_step": 408,
    "fertilized_until_day": -1,
}

In [36]:
tile = parse_tile(raw_tile)

kind_id = tile_categories[tile.kind]
crop_id = crop_categories[getattr(tile, "crop", None)]
animal_id = animal_categories[getattr(tile, "animal", None)]

In [42]:
import numpy as np
from kinds import Tile

def tile_to_ids(tile: Tile) -> np.ndarray:
    return np.array(
        [
            tile_categories[tile.kind],
            crop_categories[getattr(tile, "crop", None)],
            animal_categories[getattr(tile, "animal", None)],
        ],
        dtype=np.int64,
    )

In [43]:
tile_to_ids(tile)

array([3, 5, 0])

In [44]:
num_tile_kinds = len(TileKind)
num_crops = len(CropType) + 1
num_animals = len(AnimalType) + 1

embedding_dim = 4

In [45]:
from torch import nn

In [46]:
kind_embedding = nn.Embedding(num_tile_kinds, embedding_dim)
crops_embedding = nn.Embedding(num_crops, embedding_dim)
animal_embedding = nn.Embedding(num_animals, embedding_dim)

In [ ]:
kind_embedding

<enum 'TileKind'>

In [53]:
import torch

category_registry = {
    TileKind: tile_categories,
    CropType: crop_categories,
    AnimalType: animal_categories,
}


def get_category_id(category):
    """Seçilen enum değerinin LongTensor ID'sini döndürür."""
    try:
        category_ids = category_registry[type(category)]
    except KeyError:
        raise TypeError(
            "category bir TileKind, CropType veya AnimalType üyesi olmalı"
        ) from None

    return torch.tensor([category_ids[category]], dtype=torch.long)


get_category_id(TileKind.PLANT)
# get_category_id(CropType.MELON)
# get_category_id(AnimalType.COW)

tensor([3])

In [54]:
a_kind = get_category_id(TileKind.PLANT)

In [57]:
a_kind

tensor([3])

In [55]:
a_kind.shape

torch.Size([1])

In [56]:
kind_embedding(a_kind)

tensor([[-0.0125, -0.4860,  0.0622, -0.8008]], grad_fn=<EmbeddingBackward0>)

In [58]:
a_cow = get_category_id(AnimalType.COW)

In [59]:
a_cow

tensor([2])

In [60]:
animal_embedding(a_cow)

tensor([[-0.5745, -0.7939,  1.5517,  0.0794]], grad_fn=<EmbeddingBackward0>)

In [61]:
import torch

from kinds import parse_tile_grid
from tile_encoding import (
    NUMERIC_FEATURE_NAMES,
    encode_tile_grid,
)


observation = json_file["steps"][150]["players"][1]["observation"]

farm_index = 0
raw_tiles = observation["farms"][farm_index]["tiles"]

parsed_tiles = parse_tile_grid(raw_tiles)

categorical_ids, numeric_features = encode_tile_grid(
    tile_grid=parsed_tiles,
    current_day=observation["day"],
    current_step=observation["step"],
    market_prices=observation["market"]["prices"],
)

print("Categorical:", categorical_ids.shape)
print("Numeric:", numeric_features.shape)

KeyError: 'step'

In [65]:
observation["farms"][0]["tiles"]

[[None,
  None,
  None,
  None,
  None,
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 [None,
  None,
  {'kind': 'WEED'},
  None,
  None,
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 [None,
  None,
  {'kind': 'WEED'},
  None,
  None,
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 [None,
  None,
  {'kind': 'WEED'},
  None,
  {'kind': 'PLANT',
   'crop': 'TOMATO',
   'planted_day': 6,
   'watered_today': False,
   'consecutive_unwatered': 1,
   'yield_units': 0,
   'max_lifespan_step': -1,
   'fertilized_until_day': -1},
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 [None,
  None,
  None,
  None,
  {'kind': 'WEED'},
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 ['LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 ['LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED',
  'LOCKED'],
 ['LOCKED',


In [66]:
parse_tile_grid(observation["farms"][0]["tiles"])

[[EmptyTile(),
  EmptyTile(),
  EmptyTile(),
  EmptyTile(),
  EmptyTile(),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile()],
 [EmptyTile(),
  EmptyTile(),
  WeedTile(),
  EmptyTile(),
  EmptyTile(),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile()],
 [EmptyTile(),
  EmptyTile(),
  WeedTile(),
  EmptyTile(),
  EmptyTile(),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile()],
 [EmptyTile(),
  EmptyTile(),
  WeedTile(),
  EmptyTile(),
  PlantTile(crop=<CropType.TOMATO: 'TOMATO'>, planted_day=6, watered_today=False, consecutive_unwatered=1, yield_units=0, max_lifespan_step=-1, fertilized_until_day=-1),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile()],
 [EmptyTile(),
  EmptyTile(),
  EmptyTile(),
  EmptyTile(),
  WeedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile()],
 [LockedTile(),
  LockedTile(),
  LockedTile(),
  LockedTile(),
  Loc